# ABS Annual Government Finance Statistics 5512.0

## Python set-up

In [1]:
# analytic imports
import pandas as pd
import readabs as ra
import mgplot as mg

# pandas display settings
pd.options.display.max_rows = 9999
pd.options.display.max_columns = 999
pd.options.display.max_colwidth = 120

# display charts within this notebook
SHOW = False
RFOOTER = "ABS 5512.0"

# 5512.0 is an Excel-only release (not in the ABS Time Series Directory),
# so we hit the landing page directly.
GFS_URL = (
    "https://www.abs.gov.au/statistics/economy/government/"
    "government-finance-statistics-annual/latest-release"
)

In [2]:
CHART_DIR = "./CHARTS/5512.0 - Government Finance Statistics/"
mg.set_chart_dir(CHART_DIR)
mg.clear_chart_dir()

## Get data from ABS

In [3]:
tables = ra.grab_abs_url(url=GFS_URL)
# Identify the DO identifier prefix (changes each release, e.g. 55120DO057_202425)
do_prefixes = sorted({k.split('---')[0] for k in tables})
len(tables), do_prefixes[:3], '...', do_prefixes[-3:]

(380,
 ['55120DO001_202425', '55120DO002_202425', '55120DO003_202425'],
 '...',
 ['55120DO093_202425', '55120DO094_202425', '55120DO095_202425'])

In [4]:
# Find the DO file for a named sector by scanning each file's Contents sheet.
def find_do_prefix(sector_needle: str) -> str:
    for prefix in do_prefixes:
        contents = tables.get(f"{prefix}---Contents")
        if contents is None:
            continue
        for cell in contents.values.flatten():
            if not isinstance(cell, str):
                continue
            if sector_needle.lower() in cell.lower() and "Operating Statement" in cell:
                return prefix
    raise KeyError(f"Could not find DO file for '{sector_needle}'")


CW_PREFIX = find_do_prefix("Commonwealth Total Public Sector")
ALL_PREFIX = find_do_prefix("All Levels of Government, Total Public Sector")
CW_PREFIX, ALL_PREFIX

('55120DO057_202425', '55120DO069_202425')

## Parse Operating Statements

In [5]:
def normalize_units(u: str) -> str:
    """Map ABS short unit strings (e.g. '$m') to the long form recalibrate() understands."""
    mapping = {
        "$m": "Dollars (Millions)",
        "$M": "Dollars (Millions)",
        "$b": "Dollars (Billions)",
        "$B": "Dollars (Billions)",
        "$": "Dollars",
    }
    return mapping.get(str(u).strip(), str(u))


def parse_operating_statement(prefix: str) -> tuple[pd.DataFrame, str]:
    """Return (dataframe indexed by financial year, units). Columns are line items."""
    raw = tables[f"{prefix}---Table_1"].dropna(how='all').dropna(axis=1, how='all')
    # Row with the financial-year headers has strings like '2015-16' in non-first columns.
    header_row = None
    for idx, row in raw.iterrows():
        vals = [str(v) for v in row.values[1:] if isinstance(v, str)]
        if vals and all('-' in v and len(v) == 7 for v in vals[:3]):
            header_row = idx
            break
    if header_row is None:
        raise ValueError(f"No header row in {prefix}")
    years = raw.loc[header_row].iloc[1:].tolist()
    units_row = raw.loc[header_row + 1].iloc[1:].tolist()
    units = str(units_row[0])
    # Data rows: label in column 0, numeric values in the rest.
    data_rows = {}
    for _, row in raw.iloc[header_row + 2:].iterrows():
        label = row.iloc[0]
        if not isinstance(label, str):
            continue
        values = pd.to_numeric(row.iloc[1:], errors='coerce')
        if values.notna().any():
            data_rows[label.strip()] = values.tolist()
    df = pd.DataFrame(data_rows, index=years).T
    df.columns.name = None
    # Return with years as index, line items as columns
    out = df.T
    out.index = [pd.Period(str(int(y.split('-')[0]) + 1), freq='Y-JUN') for y in out.index]
    return out.astype(float), normalize_units(units)


cw_df, cw_units = parse_operating_statement(CW_PREFIX)
all_df, all_units = parse_operating_statement(ALL_PREFIX)
print("CW units:", cw_units, " rows:", cw_df.shape)
print("ALL units:", all_units, " rows:", all_df.shape)
cw_df.columns.tolist()

CW units: Dollars (Millions)  rows: (10, 26)
ALL units: Dollars (Millions)  rows: (10, 26)


['Taxation revenue',
 'Current grants and subsidies',
 'Sales of goods and services',
 'Interest income',
 'Other revenue',
 'Total GFS revenue',
 'Depreciation',
 'Superannuation expenses',
 'Other employee expenses',
 'Non-employee expenses',
 'Total gross operating expenses',
 'Interest on defined benefit superannuation',
 'Other interest expenses',
 'Other property expenses',
 'Grant expenses',
 'Subsidy expenses',
 'Other current transfers',
 'Capital transfer expenses',
 'Total GFS expenses',
 'GFS Net operating balance',
 'Gross fixed capital formation',
 'less Depreciation',
 'plus Change in inventories',
 'plus Other transactions in non-financial assets',
 'Total net acquisition of non-financial assets',
 'GFS NET LENDING(+)/BORROWING(-)']

## Plot headline GFS series: Commonwealth vs All Levels of Government

In [6]:
HEADLINE = [
    "Total GFS revenue",
    "Total GFS expenses",
    "GFS Net operating balance",
    "GFS NET LENDING(+)/BORROWING(-)",
]


def plot_headline(item: str) -> None:
    if item not in cw_df.columns:
        print(f"Missing in CW: {item}")
        return
    if item not in all_df.columns:
        print(f"Missing in All: {item}")
        return
    df = pd.DataFrame({
        "Commonwealth": cw_df[item],
        "All Levels of Govt": all_df[item],
    }).dropna(how='all')
    df, units = ra.recalibrate(df, cw_units)
    base = 'Australia. Total Public Sector. "All Levels" = CW + State + Local, consolidated. '
    if 'NET LENDING' in item.upper():
        base += 'Net Lending(+)/Borrowing(-) is the fiscal balance for the year. '
    mg.line_plot_finalise(
        df,
        title=f"GFS: {item}",
        ylabel=units,
        xlabel="Financial Year ending 30 June",
        width=[2, 2],
        y0=True,
        annotate=True,
        rounding=1,
        rfooter=RFOOTER,
        lfooter=base,
        legend={"loc": "best"},
        show=SHOW,
    )


for item in HEADLINE:
    plot_headline(item)

## Net Operating Balance per capita: Commonwealth and States

In [7]:
from abs_helper import get_population

STATE_NEEDLES = {
    "NSW": "New South Wales State Total Public Sector",
    "Vic.": "Victoria State Total Public Sector",
    "Qld": "Queensland State Total Public Sector",
    "SA": "South Australia State Total Public Sector",
    "WA": "Western Australia State Total Public Sector",
    "Tas.": "Tasmania State Total Public Sector",
    "NT": "Northern Territory State Total Public Sector",
    "ACT": "Australian Capital Territory State Total Public Sector",
}

state_nob = {}
for abbr, needle in STATE_NEEDLES.items():
    prefix = find_do_prefix(needle)
    df, _ = parse_operating_statement(prefix)
    state_nob[abbr] = df["GFS Net operating balance"]

ABBR_TO_FULL = dict(zip(STATE_NEEDLES.keys(), mg.state_names))


def annual_pop(state_full: str) -> pd.Series:
    """Return population at end-of-June for each year, indexed by Y-JUN Period."""
    pop, _units = get_population(state_full, project=False)
    jun = pop[pop.index.quarter == 2]
    jun.index = jun.index.asfreq("Y-JUN")
    return jun


pop_by_jurisdiction = {"CW": annual_pop("Australia")}
for abbr, full in ABBR_TO_FULL.items():
    pop_by_jurisdiction[abbr] = annual_pop(full)

nob = pd.DataFrame({
    "CW": cw_df["GFS Net operating balance"],
    **state_nob,
})
nob.shape, nob.columns.tolist()

((10, 9), ['CW', 'NSW', 'Vic.', 'Qld', 'SA', 'WA', 'Tas.', 'NT', 'ACT'])

In [8]:
def plot_nob_per_capita() -> None:
    pop_df = pd.DataFrame(pop_by_jurisdiction).reindex(nob.index)
    # NOB is in $m; multiply by 1e6 then divide by persons -> dollars per capita.
    per_capita = (nob * 1_000_000) / pop_df
    per_capita, u = ra.recalibrate(per_capita, "$")
    mg.line_plot_finalise(
        per_capita,
        title="GFS Net Operating Balance per Capita: Commonwealth and States",
        ylabel=f"{u} per person",
        xlabel="Financial Year ending 30 June",
        color=["black"] + [mg.get_color(ABBR_TO_FULL[a]) for a in STATE_NEEDLES],
        style="-",
        width=[2] + [1.5] * len(STATE_NEEDLES),
        y0=True,
        annotate=True,
        rounding=0,
        fontsize="x-small",
        rfooter=RFOOTER,
        lfooter="Australia. Total Public Sector. Per-capita using ERP at 30 June. ",
        legend={"loc": "best", "ncol": 2, "fontsize": "x-small"},
        show=SHOW,
    )


plot_nob_per_capita()

## Net Debt per capita: Commonwealth and States

In [9]:
DEBT_LIAB_ROWS = {"Currency and deposits", "Advances", "Other loans and placements", "Debt securities"}
DEBT_ASSET_ROWS = {"Currency and deposits", "Advances", "Other loans and placements"}


def net_debt_series(prefix: str) -> pd.Series:
    """Parse Balance Sheet (Table 3) and return net debt as a financial-year Series ($m)."""
    raw = tables[f"{prefix}---Table_3"].dropna(how='all').dropna(axis=1, how='all')
    # header row (same layout as the operating statement)
    header_row = None
    for idx, row in raw.iterrows():
        vals = [str(v) for v in row.values[1:] if isinstance(v, str)]
        if vals and all(len(v) == 7 and '-' in v for v in vals[:3]):
            header_row = idx
            break
    years_raw = raw.loc[header_row].iloc[1:].tolist()
    years = [pd.Period(str(int(y.split('-')[0]) + 1), freq='Y-JUN') for y in years_raw]
    # Walk rows, tracking whether we're in Assets or Liabilities block
    section = None
    liab_total = pd.Series(0.0, index=years)
    asset_total = pd.Series(0.0, index=years)
    for _, row in raw.iloc[header_row + 1:].iterrows():
        label = row.iloc[0]
        if not isinstance(label, str):
            continue
        stripped = label.strip()
        if stripped == "Assets":
            section = "asset"
            continue
        if stripped == "Liabilities":
            section = "liability"
            continue
        if stripped in ("less", "equals"):
            continue
        values = pd.to_numeric(row.iloc[1:], errors='coerce')
        if not values.notna().any():
            continue
        vals = pd.Series(values.tolist(), index=years).astype(float)
        if section == "asset" and stripped in DEBT_ASSET_ROWS:
            asset_total = asset_total.add(vals, fill_value=0)
        elif section == "liability" and stripped in DEBT_LIAB_ROWS:
            liab_total = liab_total.add(vals, fill_value=0)
    return liab_total - asset_total


net_debt = {"CW": net_debt_series(CW_PREFIX)}
for abbr, needle in STATE_NEEDLES.items():
    net_debt[abbr] = net_debt_series(find_do_prefix(needle))

nd = pd.DataFrame(net_debt)
nd.tail()

,CW,NSW,Vic.,Qld,SA,WA,Tas.,NT,ACT
2021,1305949.0,120928.0,83339.0,101893.0,25950.0,39625.0,3418.0,8005.0,5580.0
2022,1354950.0,127746.0,104414.0,117847.0,27031.0,34214.0,3025.0,9601.0,6157.0
2023,1328407.0,151295.0,122994.0,115683.0,27079.0,34452.0,3951.0,9245.0,7291.0
2024,1152406.0,163380.0,158190.0,125147.0,29616.0,36675.0,6246.0,10396.0,8997.0
2025,1184708.0,242191.0,187635.0,148149.0,33002.0,36208.0,9403.0,12144.0,11074.0


In [10]:
def plot_net_debt_per_capita() -> None:
    pop_df = pd.DataFrame(pop_by_jurisdiction).reindex(nd.index)
    per_capita = (nd * 1_000_000) / pop_df
    per_capita, u = ra.recalibrate(per_capita, "$")
    mg.line_plot_finalise(
        per_capita,
        title="GFS Net Debt per Capita: Commonwealth and States",
        ylabel=f"{u} per person",
        xlabel="Financial Year ending 30 June",
        color=["black"] + [mg.get_color(ABBR_TO_FULL[a]) for a in STATE_NEEDLES],
        style="-",
        width=[2] + [1.5] * len(STATE_NEEDLES),
        y0=True,
        annotate=True,
        rounding=0,
        fontsize="x-small",
        rfooter=RFOOTER,
        lfooter='Australia. Total Public Sector. Net Debt = interest-bearing liabilities - interest-bearing financial assets. ',
        legend={"loc": "best", "ncol": 2, "fontsize": "x-small"},
        show=SHOW,
    )


plot_net_debt_per_capita()

## Net Debt (total $): Commonwealth and States

In [11]:
def plot_net_debt_total() -> None:
    df, u = ra.recalibrate(pd.DataFrame(net_debt), "Dollars (Millions)")
    colors = ["black"] + [mg.get_color(ABBR_TO_FULL[a]) for a in STATE_NEEDLES]
    widths = [2] + [1.5] * len(STATE_NEEDLES)
    ax = mg.line_plot(
        df,
        color=colors,
        style="-",
        width=widths,
        annotate=True,
        rounding=3,
        fontsize="x-small",
    )
    ax.set_yscale("log")
    mg.finalise_plot(
        ax,
        title="GFS Net Debt: Commonwealth and States",
        ylabel=f"{u} (log scale)",
        xlabel="Financial Year ending 30 June",
        rfooter=RFOOTER,
        lfooter='Australia. Total Public Sector. Net Debt = interest-bearing liabilities - interest-bearing financial assets. ',
        legend={"loc": "best", "ncol": 2, "fontsize": "x-small"},
        show=SHOW,
    )


plot_net_debt_total()

## Net Debt to Annual Revenue: Commonwealth and States

In [12]:
# Collect Total GFS revenue for each jurisdiction (already have net_debt dict)
revenue = {"CW": cw_df["Total GFS revenue"]}
for abbr, needle in STATE_NEEDLES.items():
    df, _ = parse_operating_statement(find_do_prefix(needle))
    revenue[abbr] = df["Total GFS revenue"]
rev = pd.DataFrame(revenue)
nd_aligned = pd.DataFrame(net_debt).reindex(rev.index)
debt_to_rev = (nd_aligned / rev) * 100


def plot_debt_to_revenue() -> None:
    mg.line_plot_finalise(
        debt_to_rev,
        title="GFS Net Debt to Annual Revenue: Commonwealth and States",
        ylabel="Net Debt / Annual Revenue (%)",
        xlabel="Financial Year ending 30 June",
        color=["black"] + [mg.get_color(ABBR_TO_FULL[a]) for a in STATE_NEEDLES],
        style="-",
        width=[2] + [1.5] * len(STATE_NEEDLES),
        y0=True,
        annotate=True,
        rounding=0,
        fontsize="x-small",
        rfooter=RFOOTER,
        lfooter='Australia. Total Public Sector. State revenue includes grants from the Commonwealth. ',
        legend={"loc": "best", "ncol": 2, "fontsize": "x-small"},
        show=SHOW,
    )


plot_debt_to_revenue()
debt_to_rev.tail()

,CW,NSW,Vic.,Qld,SA,WA,Tas.,NT,ACT
2021,240.565628,129.682892,110.229482,149.016482,123.606745,56.022112,34.469544,115.345821,87.187500
2022,216.381395,117.548654,117.059991,138.767604,115.744626,47.847064,28.648546,127.030961,86.353436
2023,190.168250,134.324449,131.998970,118.459695,105.567034,46.305207,34.452389,115.779587,101.235768
2024,156.881500,137.391100,158.265968,126.536369,103.019340,44.964139,51.709579,123.555978,117.669370
2025,155.192434,185.406539,170.519916,145.144509,104.984889,40.811082,76.273524,135.945371,137.890674


## Finished

In [13]:
%load_ext watermark
%watermark -u -t -d --iversions --watermark --machine --python --conda
print("Done")

Last updated: 2026-04-21 14:21:27

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.12.0

conda environment: n/a

Compiler    : Clang 20.1.4 
OS          : Darwin
Release     : 25.3.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

mgplot : 0.2.21
pandas : 3.0.2
readabs: 0.1.8

Watermark: 2.6.0

Done
